<a href="https://colab.research.google.com/github/Zsombroo/Machine-Learning-Defense-Techniques/blob/master/Adversarial_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adversarial training

This is an example project of how to implement adversarial traning using Keras

In [0]:
# Install missing packages
!pip install cleverhans

     |████████████████████████████████| 204kB 3.4MB/s 
     |████████████████████████████████| 51kB 29.5MB/s 


In [0]:
# Import dependencies
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from cleverhans.attacks import FastGradientMethod
from cleverhans.dataset import MNIST
from cleverhans.utils_keras import KerasModelWrapper

from keras import backend as K
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint
from keras.layers import Convolution2D
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import Flatten
from keras.layers import Input
from keras.layers import MaxPooling2D
from keras.models import load_model
from keras.models import Model

tf.set_random_seed(100)
np.random.seed(100)
K.set_learning_phase(0)
sess = tf.Session()
K.set_session(sess)

W0826 12:37:40.607822 140103597946752 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/cleverhans/utils_tf.py:341: The name tf.GraphKeys is deprecated. Please use tf.compat.v1.GraphKeys instead.

Using TensorFlow backend.
W0826 12:37:40.712393 140103597946752 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:153: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.



### Prepare data

In [0]:
# Load MNIST dataset
mnist = MNIST(train_start=0, train_end=60000, test_start=0, test_end=10000)
x_train, y_train = mnist.get_set('train')
x_test, y_test = mnist.get_set('test')

### Train neural network

In [0]:
# Create convolutional neural network
input_layer = Input(shape=(28, 28, 1))
x = Convolution2D(20, (3, 3), activation='relu')(input_layer)
x = MaxPooling2D((2, 2))(x)
x = Convolution2D(20, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Convolution2D(10, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
output_layer = Dense(10, activation='softmax')(x)

target_classifier = Model(inputs=input_layer, outputs=output_layer)
target_classifier.summary()
target_classifier.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

W0826 12:37:43.737112 140103597946752 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W0826 12:37:43.746437 140103597946752 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:4138: The name tf.random_uniform is deprecated. Please use tf.random.uniform instead.

W0826 12:37:43.772459 140103597946752 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:3976: The name tf.nn.max_pool is deprecated. Please use tf.nn.max_pool2d instead.

W0826 12:37:43.837685 140103597946752 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/optimizers.py:790: The name tf.train.Optimizer is deprecated. Please use tf.compat.v1.train.Optimizer instead.

W0826 12:37:43.855974 140103597946752 deprecation_wrapper.py:119] From /us

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         (None, 28, 28, 1)         0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 26, 26, 20)        200       
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 13, 13, 20)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 11, 11, 20)        3620      
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 5, 5, 20)          0         
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 3, 3, 10)          1810      
_________________________________________________________________
max_pooling2d_3 (MaxPooling2 (None, 1, 1, 10)          0         
__________

In [0]:
# Train model
callbacks = [
    EarlyStopping(monitor='val_loss', min_delta=0.001, patience=10, verbose=1, mode='min'),
    ModelCheckpoint('weights.hdf5', monitor='val_loss', verbose=0, save_best_only=True, mode='min')
]

target_classifier.fit(x_train, y_train, batch_size=64, epochs=2000, verbose=0, callbacks=callbacks, validation_split=0.1)

W0826 12:37:43.969727 140103597946752 deprecation.py:323] From /usr/local/lib/python3.6/dist-packages/tensorflow/python/ops/math_grad.py:1250: add_dispatch_support.<locals>.wrapper (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


Epoch 00025: early stopping


In [0]:
# Load best model
target_classifier = load_model('weights.hdf5')

In [0]:
# Test model accuracy
evaluation = target_classifier.evaluate(x_test, y_test, verbose=1)
print('Loss:', evaluation[0])
print('Accuracy', evaluation[1])

10000/10000 [==============================] - 1s 57us/step
Loss: 0.058495682114828376
Accuracy 0.9832


### Generate adversarial examples

In [0]:
fgsm_parameters = {
    'eps': 0.1,
    'ord': np.inf,
    'clip_min': 0.,
    'clip_max': 1.
}

wrap = KerasModelWrapper(target_classifier)
fgsm = FastGradientMethod(wrap, sess=sess)
x_adv = fgsm.generate_np(x_test, **fgsm_parameters)

[INFO 2019-08-26 12:39:27,889 cleverhans] Constructing new graph for attack FastGradientMethod
I0826 12:39:27.889204 140103597946752 __init__.py:130] Constructing new graph for attack FastGradientMethod
W0826 12:39:27.923708 140103597946752 deprecation.py:323] From /usr/local/lib/python3.6/dist-packages/cleverhans/attacks/__init__.py:283: to_float (from tensorflow.python.ops.math_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use `tf.cast` instead.
W0826 12:39:28.047482 140103597946752 deprecation.py:506] From /usr/local/lib/python3.6/dist-packages/cleverhans/compat.py:124: calling softmax_cross_entropy_with_logits_v2_helper (from tensorflow.python.ops.nn_ops) with dim is deprecated and will be removed in a future version.
Instructions for updating:
dim is deprecated, use axis instead


In [0]:
# There are 10000 adversarial examples in X_adv
# 8000 of them are going to be used to train the model and the rest is used to
# test the accuracy before and after the training
x_adv_train = x_adv[:8000]
x_adv_test = x_adv[8000:]

In [0]:
# Test model accuracy on FGSM inputs
evaluation = target_classifier.evaluate(x_adv_test, y_test[8000:], verbose=1)
print('Loss:', evaluation[0])
print('Accuracy', evaluation[1])

2000/2000 [==============================] - 0s 49us/step
Loss: 0.8518938438892365
Accuracy 0.7485


### Adversarial training

In [0]:
# Train model
callbacks = [
    EarlyStopping(monitor='val_loss', min_delta=0.001, patience=10, verbose=1, mode='min'),
    ModelCheckpoint('adv_weights.hdf5', monitor='val_loss', verbose=0, save_best_only=True, mode='min')
]

target_classifier.fit(x_adv_train, y_test[:8000], batch_size=64, epochs=2000, verbose=0, callbacks=callbacks, validation_split=0.1)

Epoch 00020: early stopping


In [0]:
# Load best model
target_classifier = load_model('adv_weights.hdf5')

In [0]:
# Test model accuracy
evaluation = target_classifier.evaluate(x_adv_test, y_test[8000:], verbose=1)
print('Loss:', evaluation[0])
print('Accuracy', evaluation[1])

2000/2000 [==============================] - 0s 111us/step
Loss: 0.12411823223438113
Accuracy 0.9575
